# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
%run ./utils/logger

In [0]:
run_id = get_run_id()
print(run_id)

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
returns_df = spark.sql(f"""
SELECT *, (_metadata.file_name) as file_name
FROM read_files(
    'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/inbound/returns*.csv',
    format => 'csv',
    inferSchema => true
)
""")

In [0]:
display(returns_df)

In [0]:
files_recieved = returns_df.count()

if files_recieved > 0:
    print(f'Number of files recieved :{files_recieved}')
else:
    print('No files present in adls path')

In [0]:
try:
    spark.sql(f""" create table if not exists {catalog}.{schema}.returns_stage
              as
              SELECT *, (_metadata.file_name) as file_name
              FROM read_files(
                  'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/inbound/returns*.csv',format => 'csv',
                  inferSchema => true
                  )
                  """)

    log_run(run_id, "returns_ingest_pipeline", "returns_raw", "SUCCESS", "returns raw data satge load completed")

except Exception as e:
    log_run(run_id, "returns_ingest_pipeline", "returns_raw", "FAILED", error_message=str(e))
    raise 

In [0]:
%sql
describe table extended returns_stage

In [0]:
%sql
describe table extended returns

In [0]:
try:

    spark.sql(f"""
        TRUNCATE TABLE {catalog}.bronze.returns
    """)

    spark.sql(f"""
        INSERT INTO {catalog}.bronze.returns
        SELECT
            CAST(return_id AS STRING)                 AS return_id,
            CAST(order_id AS STRING)                  AS order_id,
            CAST(product_id AS STRING)                AS product_id,
            CAST(return_date AS DATE)                 AS return_date,
            CAST(refund_amount AS DECIMAL(10,2))      AS refund_amount,
            CURRENT_TIMESTAMP()                       AS ingestion_time,
            file_name                                 AS source_file
        FROM {catalog}.bronze.returns_stage
        WHERE return_id IS NOT NULL
          AND order_id IS NOT NULL
          AND product_id IS NOT NULL
    """)

    log_run(
        run_id,
        "returns_ingest_pipeline",
        "bronze_load",
        "SUCCESS",
        "Returns bronze load completed"
    )

except Exception as e:

    log_run(
        run_id,
        "returns_ingest_pipeline",
        "bronze_load",
        "FAILED",
        error_message=str(e)
    )

    raise

In [0]:
spark.sql(f'DROP TABLE commerce_raw_dev.bronze.returns_stage');